# TechMind — Preparación del dataset final (versión corregida)

### Equipo tejONEs — Laura Duque

#### 05_exploracion_dataset_final.ipynb

El proceso general que sigue este pipeline es:

- Carga de los datasets procesados de Coursera, Microsoft Learn, OpenAlex y
  Stack Exchange. Este último incorpora la corrección de usar respuestas
  aceptadas en vez de preguntas en las siete categorías.
- Validación del esquema común: `titulo`, `texto`, `categoria`, `autor` y `tipo`.
- Unificación de las fuentes y filtro a las siete categorías oficiales del
  proyecto.
- Eliminación de duplicados entre fuentes mediante el título y el texto
  normalizados.
- Verificación de disponibilidad y selección determinista, sin reemplazo, de
  200 registros por categoría.
- Auditoría de calidad y validación del total esperado de 1.400 registros.
- Exportación del dataset unificado final en la carpeta local `procesados/`.


## 1. Preparación

Se utilizan rutas locales relativas a la raíz del repositorio para leer los
datasets preparados por cada fuente: Coursera, Microsoft Learn, OpenAlex y
Stack Exchange. Para esta última fuente se usa directamente el dataset
corregido, ya traducido al español y construido con respuestas aceptadas en
vez de preguntas para las siete categorías.


In [1]:
import os
from pathlib import Path

import pandas as pd
from deep_translator import GoogleTranslator

## 2. Cargar los datasets de cada fuente

Cargo los datasets procesados de Coursera, Microsoft Learn, OpenAlex y Stack
Exchange. Para Stack Exchange se carga únicamente la versión corregida que
genera el notebook 04.


In [2]:
def find_project_root(start: Path) -> Path:
    """Encuentra la raíz del proyecto desde el directorio actual o sus padres."""
    for candidate in [start, *start.parents]:
        if (candidate / "data_science").exists() and (candidate / "README.md").exists():
            return candidate
    return start


base_dir = Path.cwd().resolve()
project_root = find_project_root(base_dir)
PROCESSED_FOLDER = str((project_root / "data_science" / "data" / "procesados").resolve())

print(f"Proyecto local: {project_root.name}")
print(f"Datos procesados: {Path(PROCESSED_FOLDER).relative_to(project_root)}")

df_coursera = pd.read_csv(Path(PROCESSED_FOLDER) / "dataset_FINAL_coursera.csv")
df_mslearn = pd.read_csv(Path(PROCESSED_FOLDER) / "dataset_FINAL_mslearn.csv")
df_openalex = pd.read_csv(Path(PROCESSED_FOLDER) / "dataset_FINAL_openalex.csv")
df_stackexchange_corregido = pd.read_csv(
    Path(PROCESSED_FOLDER) / "dataset_FINAL_stackexchange.csv"
)

print("Coursera:", len(df_coursera))
print("Microsoft Learn:", len(df_mslearn))
print("OpenAlex:", len(df_openalex))
print("Stack Exchange corregido:", len(df_stackexchange_corregido))


Proyecto local: G9-LATAM-Team-25
Datos procesados: data_science\data\procesados
Coursera: 343
Microsoft Learn: 244
OpenAlex: 463
Stack Exchange corregido: 700


## 3. Aclaración sobre la corrección del dataset de Stack Exchange

La versión anterior de Stack Exchange usaba preguntas de Stack Overflow como
texto de entrenamiento. Sin embargo, las preguntas suelen describir una duda y
no funcionan como documentación o tutorial. La corrección consiste en usar las
respuestas aceptadas, cuyo contenido explica y resuelve el problema planteado.

Aunque el alcance inicial contemplaba únicamente Mobile y Frontend, finalmente
la corrección se aplicó a las siete categorías: Backend, Bases de Datos, Cloud,
Data Science, DevOps, Frontend y Mobile.

Por este motivo, el notebook no carga ni compara el dataset anterior. Continúa
directamente con `dataset_FINAL_stackexchange.csv`, que corresponde al resultado
final corregido y traducido generado por el notebook 04.


In [3]:
# No se combina con el dataset anterior: la versión corregida cubre las siete
# categorías completas y usa respuestas aceptadas en lugar de preguntas.
print("Filas del dataset de Stack Exchange corregido:", len(df_stackexchange_corregido))
print("\nDistribución por categoría:")
print(df_stackexchange_corregido["categoria"].value_counts())


Filas del dataset de Stack Exchange corregido: 700

Distribución por categoría:
categoria
Backend           100
Bases de Datos    100
Cloud             100
Data Science      100
DevOps            100
Frontend          100
Mobile            100
Name: count, dtype: int64


## 4. Unificar todas las fuentes

Uno las cuatro fuentes en un solo dataset. Antes de buscar textos repetidos,
normalizo el contenido —todo en minúsculas y sin espacios adicionales— para
detectar duplicados aunque estén escritos de forma ligeramente distinta.

Muestro la cantidad disponible por categoría y confirmo que las siete alcancen
al menos 200 registros antes de construir el dataset final. Si alguna categoría
no llega al mínimo, el proceso se detiene con un aviso claro para evitar la
entrega de un dataset incompleto.


In [4]:
TOPE_POR_CATEGORIA = 200
CATEGORIAS_OFICIALES = ["Backend", "Bases de Datos", "Cloud", "Data Science",
                         "DevOps", "Frontend", "Mobile"]
COLUMNAS_ESPERADAS = ["titulo", "texto", "categoria", "autor", "tipo"]

# 1. Validación del esquema: confirmo que las cuatro fuentes tengan las columnas necesarias.
for nombre, df_fuente in [("Coursera", df_coursera), ("MSLearn", df_mslearn),
                           ("OpenAlex", df_openalex), ("StackExchange", df_stackexchange_corregido)]:
    faltantes = [c for c in COLUMNAS_ESPERADAS if c not in df_fuente.columns]
    if faltantes:
        raise ValueError(f"{nombre} no tiene las columnas: {faltantes}")
print("✅ Esquema validado en las cuatro fuentes.")

# 2. Uno las cuatro fuentes y conservo únicamente las siete categorías oficiales.
df_todo = pd.concat(
    [df_coursera, df_mslearn, df_openalex, df_stackexchange_corregido],
    ignore_index=True
)
antes_filtro = len(df_todo)
df_todo = df_todo[df_todo["categoria"].isin(CATEGORIAS_OFICIALES)].reset_index(drop=True)
print(f"Filas fuera de las siete categorías oficiales, descartadas: {antes_filtro - len(df_todo)}")

# 3. Normalizo el título y el texto antes de buscar duplicados.
df_todo["_titulo_normalizado"] = (
    df_todo["titulo"].str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
)
df_todo["_texto_normalizado"] = (
    df_todo["texto"].str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
)

antes = len(df_todo)
df_todo = df_todo.drop_duplicates(subset=["_titulo_normalizado", "_texto_normalizado"]).reset_index(drop=True)
print(f"Duplicados entre fuentes eliminados: {antes - len(df_todo)} (quedan {len(df_todo)})")

print("\n=== REGISTROS DISPONIBLES POR CATEGORÍA ===")
print(df_todo.groupby("categoria").size())

# 4. Confirmo que ninguna categoría quede por debajo del tope.
conteo_disponible = df_todo["categoria"].value_counts()
categorias_incompletas = conteo_disponible[conteo_disponible < TOPE_POR_CATEGORIA]

if len(categorias_incompletas) > 0:
    print("\n⚠️ ATENCIÓN: estas categorías no llegan al tope:")
    print(categorias_incompletas)
    raise ValueError("Hay categorías con menos registros de los necesarios. Revisar antes de continuar.")

# 5. Construyo el dataset final balanceado.
df_unificado = pd.concat([
    grupo.sample(min(len(grupo), TOPE_POR_CATEGORIA), random_state=42)
    for _, grupo in df_todo.groupby("categoria")
]).reset_index(drop=True)

df_unificado = df_unificado.drop(columns=["_titulo_normalizado", "_texto_normalizado"])

print("\n=== DISTRIBUCIÓN FINAL ===")
print(df_unificado["categoria"].value_counts())

assert len(df_unificado) == TOPE_POR_CATEGORIA * 7, "El total no coincide con lo esperado"
assert df_unificado["categoria"].value_counts().min() == TOPE_POR_CATEGORIA, "Alguna categoría quedó incompleta"
print("\n✅ Validación completa: 200 registros exactos en cada una de las siete categorías.")


✅ Esquema validado en las cuatro fuentes.
Filas fuera de las siete categorías oficiales, descartadas: 0
Duplicados entre fuentes eliminados: 0 (quedan 1750)

=== REGISTROS DISPONIBLES POR CATEGORÍA ===
categoria
Backend           250
Bases de Datos    250
Cloud             250
Data Science      250
DevOps            250
Frontend          250
Mobile            250
dtype: int64

=== DISTRIBUCIÓN FINAL ===
categoria
Backend           200
Bases de Datos    200
Cloud             200
Data Science      200
DevOps            200
Frontend          200
Mobile            200
Name: count, dtype: int64

✅ Validación completa: 200 registros exactos en cada una de las siete categorías.


## 5. Auditoría antes de guardar

Aquí se revisa una muestra aleatoria del resultado final para confirmar, a
simple vista, que el texto y la categoría de cada fila tengan sentido antes de
dar por terminado el dataset.


In [5]:
print("=== MUESTRA DE AUDITORÍA (15 filas al azar) ===")
df_unificado.sample(15, random_state=42)

=== MUESTRA DE AUDITORÍA (15 filas al azar) ===


,titulo,texto,categoria,autor,tipo
665,No se puede asignar una matriz con forma y tip...,Es probable que esto se deba al modo de manejo...,Data Science,Martin Brisiak,articulo
624,Transponer una matriz 1D NumPy,Está funcionando exactamente como se supone qu...,Data Science,thaking,articulo
115,Programación Java: construir un sistema de rec...,¿Alguna vez te has preguntado cómo decide Netf...,Backend,Duke University,articulo
478,Administración de datos de configuración de ap...,Obtenga información sobre los patrones de conf...,Cloud,Microsoft,módulo
233,"¿Cuál es la diferencia entre tinyint, smallint...",Ocupan diferentes cantidades de espacio y tien...,Bases de Datos,Sein Kraft,articulo
724,Administración de desafíos de seguridad de dat...,Microsoft Purview le ayuda a reforzar la segur...,Data Science,Microsoft,módulo
208,Creación de una aplicación web dinámica utiliz...,En este curso basado en proyectos de 1 hora de...,Bases de Datos,Coursera Project Network,articulo
867,Liderar la práctica de Devops y la adopción de...,"Esta investigación, realizada en organizacione...",DevOps,Krikor Maroukian,reporte
939,Entrega continua y DevOps,Amazon es famoso por entregar código nuevo cad...,DevOps,University of Virginia,articulo
479,Implementar herramientas para realizar un segu...,Obtenga información sobre cómo implementar her...,Cloud,Microsoft,módulo


## 6. Guardar los resultados finales

Guardo dos archivos en la carpeta local de datos procesados: el dataset de
Stack Exchange corregido y el dataset unificado completo con las cuatro fuentes,
listo para utilizarse en las siguientes etapas del proyecto.


In [6]:
CARPETA_PROCESADOS_ENTREGA = PROCESSED_FOLDER
os.makedirs(CARPETA_PROCESADOS_ENTREGA, exist_ok=True)

COLUMNAS_FINALES = ["titulo", "texto", "categoria", "autor", "tipo"]

df_stackexchange_corregido = df_stackexchange_corregido[COLUMNAS_FINALES]
df_unificado = df_unificado[COLUMNAS_FINALES]

ruta_stackexchange = f'{CARPETA_PROCESADOS_ENTREGA}/dataset_FINAL_stackexchange.csv'
ruta_unificado = f'{CARPETA_PROCESADOS_ENTREGA}/dataset_FINAL_UNIFICADO_techmind.csv'

df_stackexchange_corregido.to_csv(ruta_stackexchange, index=False)
df_unificado.to_csv(ruta_unificado, index=False)

print("✅ Guardado:", Path(ruta_stackexchange).name, "-", len(df_stackexchange_corregido), "filas")
print("✅ Guardado:", Path(ruta_unificado).name, "-", len(df_unificado), "filas")


✅ Guardado: dataset_FINAL_stackexchange.csv - 700 filas
✅ Guardado: dataset_FINAL_UNIFICADO_techmind.csv - 1400 filas
